 # Notebook PySpark – Análisis de Datos


---

## Tabla de Contenidos (TOC)

### Parte 1: Fundamentos y Configuración
1. [Conociendo Apache Spark](#paso-1-conociendo-apache-spark)
2. [Fundamentos de Spark para ETL y Limpieza](#paso-2-fundamentos-de-spark-para-etl-y-limpieza)
3. [Introducción a Spark vs Hadoop](#paso-3-introducción-a-spark-vs-hadoop)
4. [Diferencias entre RDDs y DataFrames](#paso-4-diferencias-entre-rdds-y-dataframes)
5. [Instalación de Spark y Anaconda en Linux](#paso-5-instalación-de-spark-y-anaconda-en-linux)
6. [Ejecución y configuración: CLI y spark-submit](#paso-6-ejecución-y-configuración-cli-y-spark-submit)
7. [Configuración de PySpark con Jupyter y Anaconda](#paso-7-configuración-de-pyspark-con-jupyter-y-anaconda)

---



## Paso 1. Conociendo Apache Spark

### ¿Qué es Apache Spark?

Apache Spark es un **motor de procesamiento distribuido** de código abierto diseñado para el análisis de grandes volúmenes de datos (Big Data). Sus características principales son:

#### Arquitectura de Spark

**1. Driver Program (Controlador)**
- Contiene la función `main()` de la aplicación
- Crea el `SparkContext` y coordina los trabajos
- Convierte el código del usuario en tareas ejecutables
- Programa las tareas en los ejecutores

**2. Cluster Manager (Administrador de Cluster)**
- Administra recursos del cluster
- Tipos: Standalone, YARN, Mesos, Kubernetes
- Asigna recursos a las aplicaciones

**3. Worker Nodes (Nodos Trabajadores)**
- Máquinas del cluster que ejecutan las tareas
- Contienen uno o más ejecutores

**4. Executors (Ejecutores)**
- Procesos JVM que ejecutan tareas en los workers
- Mantienen datos en memoria entre tareas
- Reportan estado al driver

#### Conceptos Clave

**DAG (Directed Acyclic Graph)**
- Spark construye un grafo dirigido acíclico de operaciones
- Optimiza el plan de ejecución antes de ejecutar
- Permite paralelización y optimización automática

**Lazy Evaluation (Evaluación Perezosa)**
- Las transformaciones no se ejecutan inmediatamente
- Se ejecutan solo cuando se llama una acción
- Permite optimizaciones del motor Catalyst

**Tolerancia a Fallos**
- Los RDDs mantienen información de linaje
- Puede reconstruir particiones perdidas automáticamente
- No requiere replicación costosa de datos

#### Ventajas de Spark

| Característica | Beneficio |
|----------------|-----------|
| **In-Memory Computing** | 100x más rápido que Hadoop MapReduce |
| **Facilidad de Uso** | APIs en Java, Scala, Python, R, SQL |
| **Unified Engine** | Batch, streaming, ML, graph processing |
| **Tolerancia a Fallos** | Recuperación automática sin pérdida de datos |


# Imports necesarios

In [ ]:
import os
import logging
from pathlib import Path
from typing import Optional, List, Dict, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
from pyspark.sql import Row
import time

# PySpark imports

In [ ]:
# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#%%
# Configuración global
DATA_DIR = "./data"
SRC_DIR = "./src"
OUT_DIR = "./out"

# Crear directorios si no existen
for directory in [DATA_DIR, SRC_DIR, OUT_DIR]:
    Path(directory).mkdir(exist_ok=True)
    logger.info(f"Directorio creado/verificado: {directory}")

#%%
# Funciones de utilidad
def display_df(df: DataFrame, name: str = "DataFrame", limit: int = 20) -> None:
    """
    Muestra información detallada de un DataFrame.
    
    Args:
        df: DataFrame de PySpark
        name: Nombre descriptivo del DataFrame
        limit: Número de filas a mostrar
    """
    print(f"\n{'='*50}")
    print(f"ANÁLISIS DE {name.upper()}")
    print(f"{'='*50}")
    
    print(f"\n📊 ESQUEMA:")
    df.printSchema()
    
    print(f"\n📈 ESTADÍSTICAS:")
    print(f"   • Filas: {df.count():,}")
    print(f"   • Columnas: {len(df.columns)}")
    print(f"   • Particiones: {df.rdd.getNumPartitions()}")
    
    print(f"\n📋 MUESTRA DE DATOS (primeras {limit} filas):")
    df.show(limit, truncate=False)

def with_column_if_exists(df: DataFrame, col_name: str, expression) -> DataFrame:
    """
    Añade una columna solo si no existe.
    
    Args:
        df: DataFrame de PySpark
        col_name: Nombre de la columna
        expression: Expresión para la nueva columna
    
    Returns:
        DataFrame con la nueva columna o sin cambios
    """
    if col_name not in df.columns:
        return df.withColumn(col_name, expression)
    return df

def save_dataframe(df: DataFrame, path: str, mode: str = "overwrite", 
                  format_type: str = "csv", header: bool = True) -> None:
    """
    Guarda un DataFrame con configuraciones estándar.
    
    Args:
        df: DataFrame a guardar
        path: Ruta destino
        mode: Modo de escritura (overwrite, append, etc.)
        format_type: Formato (csv, parquet, json)
        header: Incluir encabezados (solo para CSV)
    """
    try:
        writer = df.write.mode(mode)
        
        if format_type.lower() == "csv":
            writer.option("header", header).csv(path)
        elif format_type.lower() == "parquet":
            writer.parquet(path)
        elif format_type.lower() == "json":
            writer.json(path)
        else:
            raise ValueError(f"Formato no soportado: {format_type}")
            
        logger.info(f"DataFrame guardado en: {path}")
    except Exception as e:
        logger.error(f"Error al guardar DataFrame: {e}")
        raise

logger.info("Utilidades cargadas correctamente")

#%%
# Crear SparkSession global
def create_spark_session() -> SparkSession:
    """
    Crea una sesión de Spark con configuración optimizada para desarrollo local.
    
    Returns:
        SparkSession configurada
    """
    try:
        spark = SparkSession.builder \
            .appName("spark-notebook-ruta-28") \
            .master("local[*]") \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
            .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.maxResultSize", "2g") \
            .config("spark.sql.shuffle.partitions", "200") \
            .getOrCreate()
        
        # Configurar nivel de log para reducir verbosidad
        spark.sparkContext.setLogLevel("WARN")
        
        logger.info("SparkSession creada exitosamente")
        logger.info(f"Spark Version: {spark.version}")
        logger.info(f"Cores disponibles: {spark.sparkContext.defaultParallelism}")
        
        return spark
    
    except Exception as e:
        logger.error(f"Error al crear SparkSession: {e}")
        raise

In [ ]:
# Crear la sesión global
spark = create_spark_session()


## Paso 2. Fundamentos de Spark para ETL y Limpieza

### ETL (Extract, Transform, Load) con Spark

Spark proporciona herramientas poderosas para cada fase del proceso ETL:

#### Extract (Extracción)
- **Múltiples fuentes**: CSV, JSON, Parquet, bases de datos, APIs
- **Lectura optimizada**: esquemas inferidos o definidos
- **Conectores nativos**: JDBC, Kafka, S3, HDFS

#### Transform (Transformación)
- **Limpieza de datos**: nulos, duplicados, tipos inconsistentes
- **Normalización**: formatos, escalas, codificaciones
- **Agregaciones**: groupBy, window functions, UDFs
- **Joins**: inner, outer, broadcast joins optimizados

#### Load (Carga)
- **Múltiples destinos**: archivos, bases de datos, data lakes
- **Modos de escritura**: overwrite, append, errorIfExists
- **Optimizaciones**: particionado, compresión, columnar



### Ejemplo Práctico: Dataset de Clientes

Vamos a crear un dataset sintético con problemas comunes de calidad de datos:

In [ ]:

# Crear dataset sintético con problemas de calidad
def create_synthetic_customer_data():
    """
    Crea un dataset sintético de clientes con problemas típicos de calidad de datos.
    """
    
    # Datos con problemas intencionados
    customers_data = [
        (1, "Juan Pérez", "juan.perez@email.com", "28", "Madrid", "España", "2023-01-15"),
        (2, "María García", "MARIA.GARCIA@GMAIL.COM", "35", "Barcelona", "España", "2023-02-20"),
        (3, "Carlos López", "", "45", "Valencia", "España", "2023-03-10"),  # Email vacío
        (4, "Ana Rodríguez", "ana@email.com", "treinta", "Sevilla", "España", "2023-04-05"),  # Edad texto
        (5, "Pedro Martín", "pedro.martin@email.com", "28", "Madrid", "España", "2023-01-15"),  # Duplicado
        (6, "", "lucia@email.com", "32", "Bilbao", "España", "2023-05-12"),  # Nombre vacío
        (7, "David Santos", "david@email.com", None, "Granada", "España", "2023-06-18"),  # Edad nula
        (8, "Laura Jiménez", "laura.jimenez@email.com", "42", "", "España", "2023-07-22"),  # Ciudad vacía
        (9, "Miguel Torres", "miguel.torres@email.com", "29", "Zaragoza", None, "2023-08-14"),  # País nulo
        (10, "Carmen Ruiz", "carmen@email.com", "38", "Málaga", "España", ""),  # Fecha vacía
    ]
    
    # Definir esquema explícito
    schema = StructType([
        StructField("customer_id", IntegerType(), False),
        StructField("name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("age", StringType(), True),  # Intencionalmente como String
        StructField("city", StringType(), True),
        StructField("country", StringType(), True),
        StructField("registration_date", StringType(), True)
    ])
    
    # Crear DataFrame
    df = spark.createDataFrame(customers_data, schema)
    
    # Guardar como CSV
    csv_path = f"{DATA_DIR}/customers.csv"
    save_dataframe(df, csv_path, format_type="csv")
    
    logger.info(f"Dataset sintético creado: {csv_path}")
    return df

In [ ]:


# Crear el dataset
customers_df = create_synthetic_customer_data()
display_df(customers_df, "Customers (Raw Data)")


### Diccionario de Datos - Dataset Customers

| Campo | Tipo Original | Descripción | Problemas Detectados |
|-------|---------------|-------------|---------------------|
| `customer_id` | Integer | Identificador único del cliente | ✓ Sin problemas |
| `name` | String | Nombre completo del cliente | ❌ Valores vacíos |
| `email` | String | Dirección de correo electrónico | ❌ Valores vacíos, formatos inconsistentes |
| `age` | String | Edad del cliente (debería ser Integer) | ❌ Valores de texto, nulos |
| `city` | String | Ciudad de residencia | ❌ Valores vacíos |
| `country` | String | País de residencia | ❌ Valores nulos |
| `registration_date` | String | Fecha de registro (formato YYYY-MM-DD) | ❌ Valores vacíos |


In [ ]:

# Proceso de limpieza y transformación ETL
def clean_customer_data(df: DataFrame) -> DataFrame:
    """
    Limpia y transforma el dataset de clientes.
    
    Args:
        df: DataFrame crudo de clientes
        
    Returns:
        DataFrame limpio y transformado
    """
    logger.info("Iniciando proceso de limpieza de datos...")
    
    # 1. Identificar registros con problemas
    print("📊 ANÁLISIS DE CALIDAD INICIAL:")
    df.select([
        count(when(col(c).isNull() | (col(c) == ""), c)).alias(f"null_empty_{c}")
        for c in df.columns
    ]).show()
    
    # 2. Limpiar emails: normalizar a minúsculas y validar formato básico
    df_clean = df.withColumn(
        "email_clean", 
        when(
            col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"),
            lower(trim(col("email")))
        ).otherwise(None)
    )
    
    # 3. Limpiar edad: convertir a entero, manejar valores inválidos
    df_clean = df_clean.withColumn(
        "age_clean",
        when(
            col("age").rlike(r"^\d+$") & (col("age").cast("integer").between(0, 120)),
            col("age").cast("integer")
        ).otherwise(None)
    )
    
    # 4. Limpiar nombres: remover espacios y valores vacíos
    df_clean = df_clean.withColumn(
        "name_clean",
        when(
            (trim(col("name")) != "") & col("name").isNotNull(),
            trim(col("name"))
        ).otherwise(None)
    )
    
    # 5. Limpiar ciudad y país
    df_clean = df_clean.withColumn(
        "city_clean",
        when(
            (trim(col("city")) != "") & col("city").isNotNull(),
            trim(col("city"))
        ).otherwise("No Especificada")
    )
    
    df_clean = df_clean.withColumn(
        "country_clean",
        when(
            (trim(col("country")) != "") & col("country").isNotNull(),
            trim(col("country"))
        ).otherwise("No Especificado")
    )
    
    # 6. Limpiar fecha: convertir a formato date
    df_clean = df_clean.withColumn(
        "registration_date_clean",
        when(
            col("registration_date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            to_date(col("registration_date"), "yyyy-MM-dd")
        ).otherwise(None)
    )
    
    # 7. Añadir columnas de calidad
    df_clean = df_clean.withColumn(
        "data_quality_score",
        (
            when(col("name_clean").isNotNull(), 1).otherwise(0) +
            when(col("email_clean").isNotNull(), 1).otherwise(0) +
            when(col("age_clean").isNotNull(), 1).otherwise(0) +
            when(col("registration_date_clean").isNotNull(), 1).otherwise(0)
        ).cast("double") / 4 * 100
    )
    
    # 8. Seleccionar columnas finales
    df_final = df_clean.select(
        col("customer_id"),
        col("name_clean").alias("name"),
        col("email_clean").alias("email"),
        col("age_clean").alias("age"),
        col("city_clean").alias("city"),
        col("country_clean").alias("country"),
        col("registration_date_clean").alias("registration_date"),
        col("data_quality_score")
    )
    
    logger.info("Proceso de limpieza completado")
    return df_final

In [ ]:
# Aplicar limpieza
customers_clean = clean_customer_data(customers_df)
display_df(customers_clean, "Customers (Clean Data)")

In [ ]:

# Análisis de calidad post-limpieza
print("\n📊 ANÁLISIS DE CALIDAD POST-LIMPIEZA:")

# Estadísticas de calidad
quality_stats = customers_clean.agg(
    avg("data_quality_score").alias("avg_quality_score"),
    min("data_quality_score").alias("min_quality_score"),
    max("data_quality_score").alias("max_quality_score"),
    count("*").alias("total_records")
).collect()[0]

print(f"   • Registros totales: {quality_stats['total_records']}")
print(f"   • Calidad promedio: {quality_stats['avg_quality_score']:.1f}%")
print(f"   • Calidad mínima: {quality_stats['min_quality_score']:.1f}%")
print(f"   • Calidad máxima: {quality_stats['max_quality_score']:.1f}%")

# Distribución por calidad
print("\n📈 DISTRIBUCIÓN POR NIVEL DE CALIDAD:")
customers_clean.groupBy("data_quality_score") \
    .count() \
    .orderBy("data_quality_score") \
    .show()

### Técnicas Avanzadas de Limpieza

#### Deduplicación
Eliminación de registros duplicados basada en criterios específicos:


In [ ]:

#%%
# Deduplicación avanzada
def deduplicate_customers(df: DataFrame) -> DataFrame:
    """
    Elimina duplicados manteniendo el registro de mejor calidad.
    """
    # Buscar duplicados potenciales por email
    duplicates = df.filter(col("email").isNotNull()) \
        .groupBy("email") \
        .count() \
        .filter(col("count") > 1)
    
    print(f"📊 Emails duplicados encontrados: {duplicates.count()}")
    
    if duplicates.count() > 0:
        duplicates.show()
        
        # Para duplicados, mantener el de mejor calidad
        window_spec = Window.partitionBy("email").orderBy(
            col("data_quality_score").desc(),
            col("customer_id").asc()
        )
        
        df_deduplicated = df.withColumn(
            "row_number", 
            row_number().over(window_spec)
        ).filter(col("row_number") == 1) \
         .drop("row_number")
        
        logger.info("Deduplicación completada")
        return df_deduplicated
    
    return df

customers_final = deduplicate_customers(customers_clean)
display_df(customers_final, "Customers (Final)")

### Validaciones y Controles de Calidad

In [ ]:

# Validaciones finales
def validate_data_quality(df: DataFrame) -> bool:
    """
    Ejecuta validaciones de calidad de datos.
    """
    print("🔍 EJECUTANDO VALIDACIONES DE CALIDAD...")
    
    # Validación 1: No debe haber customer_ids duplicados
    unique_ids = df.select("customer_id").distinct().count()
    total_records = df.count()
    
    assert unique_ids == total_records, f"IDs duplicados: {total_records - unique_ids}"
    print("   ✓ No hay customer_ids duplicados")
    
    # Validación 2: Edades deben estar en rango válido
    invalid_ages = df.filter(
        col("age").isNotNull() & 
        ~col("age").between(0, 120)
    ).count()
    
    assert invalid_ages == 0, f"Edades inválidas encontradas: {invalid_ages}"
    print("   ✓ Todas las edades están en rango válido")
    
    # Validación 3: Emails deben tener formato válido
    invalid_emails = df.filter(
        col("email").isNotNull() & 
        ~col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
    ).count()
    
    assert invalid_emails == 0, f"Emails con formato inválido: {invalid_emails}"
    print("   ✓ Todos los emails tienen formato válido")
    
    # Validación 4: Score de calidad mínimo
    low_quality = df.filter(col("data_quality_score") < 50).count()
    print(f"   ⚠️ Registros con calidad < 50%: {low_quality}")
    
    print("✅ TODAS LAS VALIDACIONES PASARON")
    return True


In [ ]:

# Ejecutar validaciones
validate_data_quality(customers_final)

# Guardar dataset final limpio
save_dataframe(customers_final, f"{OUT_DIR}/customers_clean", format_type="parquet")


## Paso 3. Introducción a Spark vs Hadoop

### Comparación Arquitectural

#### Hadoop MapReduce
- **Paradigma**: Solo batch processing
- **Almacenamiento**: Disco (HDFS) en cada operación
- **Modelo**: Map → Shuffle → Reduce
- **Velocidad**: Lento por I/O intensivo
- **Tolerancia a fallos**: Replicación de datos (3x por defecto)

#### Apache Spark
- **Paradigma**: Batch, streaming, ML, graph processing
- **Almacenamiento**: In-memory con spillover a disco
- **Modelo**: DAG optimizado con lazy evaluation
- **Velocidad**: 100x más rápido en memoria, 10x en disco
- **Tolerancia a fallos**: Linaje de RDDs (recomputation)

### Tabla Comparativa Detallada

| Aspecto | Hadoop MapReduce | Apache Spark |
|---------|------------------|--------------|
| **Velocidad** | Lento (disco I/O) | Muy rápido (in-memory) |
| **Facilidad de uso** | Complejo (Java/Python) | Simple (Python/Scala/SQL) |
| **Procesamiento** | Solo Batch | Batch + Streaming + ML |
| **Memoria** | Basado en disco | In-memory + disco |
| **Tolerancia a fallos** | Replicación | Linaje RDD |
| **Curva de aprendizaje** | Steep | Moderada |
| **Ecosistema** | Maduro | En crecimiento |
| **Costo** | Menor (hardware) | Mayor (RAM) |

### ¿Cuándo usar cada uno?

#### Usar Hadoop MapReduce cuando:
- **Presupuesto limitado** para hardware (RAM costosa)
- **Datos muy grandes** que no caben en memoria del cluster
- **Operaciones simples** de ETL batch
- **Ecosistema Hadoop** ya establecido (HBase, Hive, etc.)
- **Estabilidad** es más importante que velocidad

#### Usar Apache Spark cuando:
- **Velocidad** es crítica
- **Análisis iterativos** (ML, graph algorithms)
- **Streaming en tiempo real** requerido
- **Desarrollo ágil** con APIs amigables
- **Procesamiento complejo** con múltiples pasos
- **Recursos suficientes** de memoria



### Ejemplo Comparativo: WordCount


### Simulación de Comparación de Rendimiento

Vamos a simular cómo se vería el mismo proceso en ambas plataformas:


In [ ]:

# Crear datos de prueba para comparación
def create_text_data():
    """
    Crea datos de texto para comparar MapReduce vs Spark.
    """
    text_data = [
        "Apache Spark es un motor de procesamiento distribuido",
        "Hadoop MapReduce procesa datos en lotes",
        "Spark puede procesar datos en memoria",
        "MapReduce escribe resultados intermedios a disco",
        "Spark optimiza el DAG antes de ejecutar",
        "MapReduce tiene alta latencia por I/O",
        "Spark soporta streaming y batch processing",
        "Hadoop tiene un ecosistema maduro y estable"
    ] * 1000  # Repetir para simular volumen

    # Crear RDD
    text_rdd = spark.sparkContext.parallelize(text_data, numSlices=4)
    
    # Crear DataFrame
    text_df = spark.createDataFrame(
        [(i, line) for i, line in enumerate(text_data)],
        ["id", "text"]
    )
    
    return text_rdd, text_df

text_rdd, text_df = create_text_data()

#%%
# Simulación estilo MapReduce (con RDDs)

def mapreduce_style_wordcount(rdd):
    """
    WordCount estilo MapReduce usando RDDs.
    """
    start_time = time.time()
    
    # Map: dividir en palabras
    words_rdd = rdd.flatMap(lambda line: line.lower().split())
    
    # Map: crear pares (palabra, 1)
    word_pairs_rdd = words_rdd.map(lambda word: (word, 1))
    
    # Reduce: sumar ocurrencias
    word_counts_rdd = word_pairs_rdd.reduceByKey(lambda a, b: a + b)
    
    # Recolectar resultados (acción)
    results = word_counts_rdd.collect()
    
    end_time = time.time()
    
    print(f"⏱️ MapReduce-style (RDD): {end_time - start_time:.4f} segundos")
    print(f"📊 Total palabras únicas: {len(results)}")
    
    # Mostrar top 10
    top_words = sorted(results, key=lambda x: x[1], reverse=True)[:10]
    print("🔝 Top 10 palabras:")
    for word, count in top_words:
        print(f"   {word}: {count}")
    
    return word_counts_rdd

mapreduce_result = mapreduce_style_wordcount(text_rdd)

#%%
# Estilo Spark optimizado (con DataFrames)
def spark_style_wordcount(df):
    """
    WordCount estilo Spark usando DataFrames y SQL.
    """
    start_time = time.time()
    
    # Usar funciones de Spark SQL
    word_counts_df = df.select(
        explode(split(lower(col("text")), " ")).alias("word")
    ).filter(
        col("word") != ""
    ).groupBy("word") \
     .count() \
     .orderBy(col("count").desc())
    
    # Mostrar plan de ejecución optimizado
    print("📋 PLAN DE EJECUCIÓN OPTIMIZADO:")
    word_counts_df.explain(True)
    
    # Ejecutar y recolectar
    results = word_counts_df.collect()
    
    end_time = time.time()
    
    print(f"⏱️ Spark-style (DataFrame): {end_time - start_time:.4f} segundos")
    print(f"📊 Total palabras únicas: {len(results)}")
    
    # Mostrar top 10
    print("🔝 Top 10 palabras:")
    word_counts_df.show(10, truncate=False)
    
    return word_counts_df

spark_result = spark_style_wordcount(text_df)


### Optimizaciones de Spark vs Limitaciones de MapReduce

#### Optimizaciones Catalyst (Spark)
- **Predicate Pushdown**: Filtros se ejecutan antes de joins
- **Projection Pushdown**: Solo columnas necesarias se leen
- **Constant Folding**: Expresiones constantes se evalúan una vez
- **Boolean Expression Simplification**: Optimización de condiciones

#### Limitaciones MapReduce
- **Sin optimización automática**: El desarrollador debe optimizar
- **I/O intensivo**: Cada job escribe a disco
- **Setup overhead**: JVM startup para cada task
- **No reutilización de datos**: No sharing entre jobs


## Paso 4. Diferencias entre RDDs y DataFrames

### Conceptos Fundamentales

#### RDDs (Resilient Distributed Datasets)
- **Abstracción fundamental** de Spark
- **Tipado fuerte** (compile-time type safety)
- **API funcional** (map, filter, reduce)
- **Sin optimización automática**
- **Control total** sobre distribución y particionado

#### DataFrames
- **Abstracción de alto nivel** sobre RDDs
- **Esquema estructurado** (columnas con tipos)
- **Optimización automática** (Catalyst optimizer)
- **API SQL-like** más intuitiva
- **Compatibilidad multi-lenguaje**

### Tabla Comparativa Completa

| Aspecto | RDDs | DataFrames |
|---------|------|------------|
| **Nivel de abstracción** | Bajo nivel | Alto nivel |
| **Tipado** | Fuerte (compile-time) | Débil (runtime) |
| **Optimización** | Manual | Automática (Catalyst) |
| **Performance** | Depende del código | Optimizada |
| **Facilidad de uso** | Compleja | Simple |
| **Debugging** | Fácil | Más difícil |
| **Serialización** | Java/Kryo | Tungsten (optimizada) |
| **Compatibilidad** | Scala/Java mejor | Todos los lenguajes |


### Motor Catalyst y Tungsten

#### Catalyst Optimizer
- **Rule-based optimization**: Aplica reglas predefinidas
- **Cost-based optimization**: Elige el plan de menor costo
- **Code generation**: Genera código Java optimizado
- **Estadísticas de columnas**: Para mejores decisiones

#### Tungsten Execution Engine
- **Off-heap memory management**: Reduce GC pressure
- **Cache-aware computation**: Optimizado para CPU caches
- **Code generation**: Elimina overhead de interpretación
- **Columnar storage**: Mejor compresión y acceso

In [ ]:
# Ejemplos comparativos: RDD vs DataFrame
def rdd_vs_dataframe_examples():
    """
    Ejemplos lado a lado de RDD vs DataFrame para operaciones comunes.
    """
    
    # Datos de ejemplo: información de empleados
    employees_data = [
        (1, "Juan", "IT", 50000, 28),
        (2, "María", "Marketing", 55000, 32),
        (3, "Carlos", "IT", 60000, 35),
        (4, "Ana", "HR", 45000, 29),
        (5, "Pedro", "IT", 65000, 40),
        (6, "Laura", "Marketing", 52000, 27),
        (7, "David", "Finance", 70000, 45),
        (8, "Carmen", "IT", 58000, 33)
    ]
    
    print("🔄 COMPARACIÓN RDD vs DATAFRAME")
    print("="*60)
    
    # ========== CREACIÓN ==========
    print("\n1️⃣ CREACIÓN DE DATOS")
    
    # RDD approach
    print("\n📊 RDD Approach:")
    employees_rdd = spark.sparkContext.parallelize(employees_data)
    print(f"   • Tipo: {type(employees_rdd)}")
    print(f"   • Particiones: {employees_rdd.getNumPartitions()}")
    print(f"   • Muestra: {employees_rdd.take(2)}")
    
    # DataFrame approach
    print("\n📊 DataFrame Approach:")
    schema = StructType([
        StructField("id", IntegerType(), False),
        StructField("name", StringType(), False),
        StructField("department", StringType(), False),
        StructField("salary", IntegerType(), False),
        StructField("age", IntegerType(), False)
    ])
    
    employees_df = spark.createDataFrame(employees_data, schema)
    print(f"   • Tipo: {type(employees_df)}")
    print(f"   • Esquema definido: ✓")
    employees_df.printSchema()
    
    # ========== FILTRADO ==========
    print("\n2️⃣ FILTRADO (Empleados IT con salario > 55000)")
    
    # RDD approach
    print("\n📊 RDD Approach:")
    rdd_filtered = employees_rdd.filter(
        lambda emp: emp[2] == "IT" and emp[3] > 55000
    )
    rdd_result = rdd_filtered.collect()
    print(f"   • Resultados: {len(rdd_result)}")
    for emp in rdd_result:
        print(f"     {emp}")
    
    # DataFrame approach
    print("\n📊 DataFrame Approach:")
    df_filtered = employees_df.filter(
        (col("department") == "IT") & (col("salary") > 55000)
    )
    print(f"   • Plan optimizado disponible: ✓")
    df_filtered.show(truncate=False)
    
    # ========== AGREGACIONES ==========
    print("\n3️⃣ AGREGACIONES (Salario promedio por departamento)")
    
    # RDD approach
    print("\n📊 RDD Approach:")
    dept_salaries_rdd = employees_rdd.map(lambda emp: (emp[2], emp[3]))
    avg_salary_rdd = dept_salaries_rdd.aggregateByKey(
        (0, 0),  # (suma, count)
        lambda acc, value: (acc[0] + value, acc[1] + 1),
        lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    ).mapValues(lambda x: x[0] / x[1])
    
    rdd_avg_result = avg_salary_rdd.collect()
    print("   • Salarios promedio:")
    for dept, avg in sorted(rdd_avg_result):
        print(f"     {dept}: ${avg:,.2f}")
    
    # DataFrame approach
    print("\n📊 DataFrame Approach:")
    df_avg = employees_df.groupBy("department") \
        .agg(avg("salary").alias("avg_salary")) \
        .orderBy("department")
    
    print("   • Salarios promedio (optimizado):")
    df_avg.show(truncate=False)
    
    # ========== PERFORMANCE COMPARISON ==========
    print("\n4️⃣ ANÁLISIS DE PERFORMANCE")
    
    # Crear dataset más grande para medir diferencias
    large_data = employees_data * 10000  # 80,000 registros
    
    # RDD timing
    large_rdd = spark.sparkContext.parallelize(large_data, numSlices=8)
    start_time = time.time()
    rdd_count = large_rdd.filter(lambda emp: emp[3] > 50000).count()
    rdd_time = time.time() - start_time
    
    # DataFrame timing
    large_df = spark.createDataFrame(large_data, schema)
    start_time = time.time()
    df_count = large_df.filter(col("salary") > 50000).count()
    df_time = time.time() - start_time
    
    print(f"\n📊 Filtrado en dataset de {len(large_data):,} registros:")
    print(f"   • RDD: {rdd_time:.4f}s (resultado: {rdd_count:,})")
    print(f"   • DataFrame: {df_time:.4f}s (resultado: {df_count:,})")
    print(f"   • Mejora: {rdd_time/df_time:.2f}x más rápido con DataFrame")
    
    return employees_rdd, employees_df

employees_rdd, employees_df = rdd_vs_dataframe_examples()



### Cuándo Usar RDDs vs DataFrames

#### Usar RDDs cuando:
- **Manipulación de datos no estructurados** (texto, binarios)
- **Control fine-grained** sobre distribución de datos
- **Operaciones de bajo nivel** no disponibles en DataFrames
- **Compatibilidad** con código legacy
- **Debugging detallado** es necesario

#### Usar DataFrames cuando:
- **Datos estructurados** o semi-estructurados
- **Performance** es crítica
- **Desarrollo rápido** requerido
- **Integraciones SQL** necesarias
- **Equipos con experiencia SQL** limitada en programación funcional


In [ ]:

# Ejemplo avanzado: Conversión entre RDD y DataFrame
def rdd_dataframe_conversion():
    """
    Demuestra conversión bidireccional entre RDD y DataFrame.
    """
    print("🔄 CONVERSIÓN ENTRE RDD Y DATAFRAME")
    print("="*50)
    
    # DataFrame a RDD
    print("\n1️⃣ DataFrame → RDD")
    df_to_rdd = employees_df.rdd
    print(f"   • Tipo original: {type(employees_df)}")
    print(f"   • Tipo convertido: {type(df_to_rdd)}")
    print(f"   • Muestra: {df_to_rdd.take(2)}")
    
    # RDD a DataFrame (con esquema)
    print("\n2️⃣ RDD → DataFrame (con esquema)")
    rdd_to_df = spark.createDataFrame(employees_rdd, schema)
    print(f"   • Tipo original: {type(employees_rdd)}")
    print(f"   • Tipo convertido: {type(rdd_to_df)}")
    rdd_to_df.show(3)
    
    # RDD a DataFrame (con inferencia)
    print("\n3️⃣ RDD → DataFrame (esquema inferido)")
    # Crear RDD con Row objects
    row_rdd = employees_rdd.map(lambda x: Row(
        id=x[0], name=x[1], department=x[2], salary=x[3], age=x[4]
    ))
    
    inferred_df = spark.createDataFrame(row_rdd)
    print("   • Esquema inferido:")
    inferred_df.printSchema()
    
    return df_to_rdd, rdd_to_df, inferred_df


In [ ]:
rdd_converted, df_converted, df_inferred = rdd_dataframe_conversion()


## Paso 5. Instalación de Spark y Anaconda en Linux

### Requisitos del Sistema

#### Hardware Mínimo
- RAM: 8GB mínimo (16GB recomendado)
- CPU: 4 cores mínimo
- Disco: 20GB libre
- Red: Conexión a Internet para descargas

#### Software Base
- SO: Ubuntu 20.04 LTS o superior
- Java: OpenJDK 11
- Python: 3.8+ (vía Anaconda)

### Proceso de Instalación

1. **Actualizar Sistema**
```     
sudo apt update && sudo apt upgrade -y